# Setup ENSEMBL annotations
##
### Author: Martin Loza
### Date: 25/11/20

Let's setup the ENSEMBL annotation for a given species for further use

In [460]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
})

# Local variables
seed = 777
date = "251120"

species = "zebrafish"
# "armadillo" "chicken" "dog" "human" "drosophila" "elegans" "ferret" "macaque" "mouse" "rat" "zebrafish"
in_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/raw/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"

### Load the data

In [461]:
# Load the transcripts data
# Search for the file related to the species
files <- list.files(in_dir)
file_name <- files[grep(species, files)]
# paste0(in_dir,file_name)
# Load the transcripts
transcripts = read.table(paste0(in_dir,file_name), sep = "\t", header = TRUE, 
                         stringsAsFactors = FALSE, quote = "", comment.char = "", 
                         fill = TRUE )
head(transcripts,3)
colnames(transcripts)
cat("Number of transcripts loaded:", nrow(transcripts), "\n")


,Gene.stable.ID,Gene.stable.ID.version,Transcript.stable.ID,Transcript.stable.ID.version,Gene.description,Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Transcript.start..bp.,Transcript.end..bp.,Transcription.start.site..TSS.,Gene.name,Transcript.name,Gene...GC.content,Gene.type,Transcript.type,Version..gene.,Version..transcript.
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>
1,ENSDARG00000063344,ENSDARG00000063344.8,ENSDART00000131829,ENSDART00000131829.2,family with sequence similarity 162 member A [Source:NCBI gene (formerly Entrezgene);Acc:336363],24,20927135,20934666,1,20927202,20934666,20927202,fam162a,fam162a-201,32.65,protein_coding,protein_coding,8,2
2,ENSDARG00000063344,ENSDARG00000063344.8,ENSDART00000144883,ENSDART00000144883.3,family with sequence similarity 162 member A [Source:NCBI gene (formerly Entrezgene);Acc:336363],24,20927135,20934666,1,20927135,20934666,20927135,fam162a,fam162a-202,32.65,protein_coding,protein_coding,8,3
3,ENSDARG00000097685,ENSDARG00000097685.3,ENSDART00000156963,ENSDART00000156963.2,,17,50460321,50472985,-1,50460321,50472985,50472985,si:ch211-235i11.3,si:ch211-235i11.3-201,41.89,protein_coding,protein_coding,3,2


[1] "Gene.stable.ID"                 "Gene.stable.ID.version"        
 [3] "Transcript.stable.ID"           "Transcript.stable.ID.version"  
 [5] "Gene.description"               "Chromosome.scaffold.name"      
 [7] "Gene.start..bp."                "Gene.end..bp."                 
 [9] "Strand"                         "Transcript.start..bp."         
[11] "Transcript.end..bp."            "Transcription.start.site..TSS."
[13] "Gene.name"                      "Transcript.name"               
[15] "Gene...GC.content"              "Gene.type"                     
[17] "Transcript.type"                "Version..gene."                
[19] "Version..transcript."

Number of transcripts loaded: 65905 


In [462]:
unique(transcripts$Transcript.type) 

[1] "protein_coding"                     "processed_transcript"              
 [3] "nonsense_mediated_decay"            "retained_intron"                   
 [5] "lincRNA"                            "antisense"                         
 [7] "sense_intronic"                     "TR_J_gene"                         
 [9] "unprocessed_pseudogene"             "processed_pseudogene"              
[11] "transcribed_unprocessed_pseudogene" "TEC"                               
[13] "sense_overlapping"                  "TR_V_gene"                         
[15] "IG_C_gene"                          "polymorphic_pseudogene"            
[17] "pseudogene"                         "TR_D_gene"                         
[19] "TR_V_pseudogene"                    "rRNA"                              
[21] "miRNA"                              "non_stop_decay"                    
[23] "IG_V_pseudogene"                    "IG_C_pseudogene"                   
[25] "IG_J_pseudogene"                    "IG_pseudogene"                     
[27] "scaRNA"                             "snRNA"                             
[29] "snoRNA"                             "misc_RNA"                          
[31] "ribozyme"                           "sRNA"                              
[33] "Mt_tRNA"                            "Mt_rRNA"

### Handle transcript versions and create final annotations

We need to:
1. Extract transcript versions from the Transcript.stable.ID.version column
2. Keep only the highest version for each transcript ID
3. Create separate gene and transcript annotation files

#### 1. Extract transcript versions 

In [463]:
# We first confirm that the Transcript.stable.ID.version column exists
if(!"Transcript.stable.ID.version" %in% colnames(transcripts)) {
    warning("The column 'Transcript.stable.ID.version' does not exist in the transcripts data.\n Using Transcript.stable.ID' instead.")
    transcripts$Transcript.stable.ID.version <- transcripts$Transcript.stable.ID
}

# Extract version number from the transcript ID
transcripts$transcript_version <- as.numeric(str_extract(transcripts$Transcript.stable.ID.version, "(?<=\\.)\\d+$"))

# Check for any NA values in transcript_version
cat("Number of transcripts with missing version:", sum(is.na(transcripts$transcript_version)), "\n")

# Show some examples of version extraction
cat("Examples of transcript ID and extracted versions:\n")
sample_idx <- sample(nrow(transcripts), 3)
print(transcripts[sample_idx, c("Transcript.stable.ID", "Transcript.stable.ID.version", "transcript_version")])

# Check the range of versions
cat("\nTranscript version summary:\n")
print(summary(transcripts$transcript_version))


Number of transcripts with missing version: 0 
Examples of transcript ID and extracted versions:
      Transcript.stable.ID Transcript.stable.ID.version transcript_version
12146   ENSDART00000080632         ENSDART00000080632.4                  4
20834   ENSDART00000138469         ENSDART00000138469.2                  2
34293   ENSDART00000135221         ENSDART00000135221.2                  2

Transcript version summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  1.000   2.000   2.000   2.812   3.000  13.000 


#### 2. Keep only the highest transcript versions

In [464]:
# check if the transcript_version columns exists
if(!"transcript_version" %in% colnames(transcripts)) {
    stop("The column 'transcript_version' does not exist in the transcripts data.")
} else{
    # Keep only the highest version for each transcript ID
    cat("Original number of transcripts:", nrow(transcripts), "\n")

    # Group by transcript stable ID and keep only the row with the highest version
    transcripts_filtered <- transcripts %>%
        group_by(Transcript.stable.ID) %>%
        slice_max(transcript_version, n = 1, with_ties = FALSE) %>%
        ungroup()

    cat("Number of transcripts after keeping highest versions:", nrow(transcripts_filtered), "\n")
    cat("Number of transcripts removed:", nrow(transcripts) - nrow(transcripts_filtered), "\n")

    # Check if we have any duplicated transcript IDs (should be 0)
    cat("Number of duplicated transcript IDs after filtering:", sum(duplicated(transcripts_filtered$Transcript.stable.ID)), "\n")
}
rm(transcripts) # Clean up the original transcripts to avoid confusion and free memory
gc()

Original number of transcripts: 65905 
Number of transcripts after keeping highest versions: 65905 
Number of transcripts removed: 0 
Number of duplicated transcript IDs after filtering: 0 


,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,1319939,70.5,4066685,217.2,NA,6611503,353.1
Vcells,7835607,59.8,25698066,196.1,65536,52047127,397.1


Let's focus on protein coding genes and typical ncRNA genes

In [465]:
colnames(transcripts_filtered)

[1] "Gene.stable.ID"                 "Gene.stable.ID.version"        
 [3] "Transcript.stable.ID"           "Transcript.stable.ID.version"  
 [5] "Gene.description"               "Chromosome.scaffold.name"      
 [7] "Gene.start..bp."                "Gene.end..bp."                 
 [9] "Strand"                         "Transcript.start..bp."         
[11] "Transcript.end..bp."            "Transcription.start.site..TSS."
[13] "Gene.name"                      "Transcript.name"               
[15] "Gene...GC.content"              "Gene.type"                     
[17] "Transcript.type"                "Version..gene."                
[19] "Version..transcript."           "transcript_version"

In [466]:
unique(transcripts_filtered$Transcript.type)

[1] "protein_coding"                     "nonsense_mediated_decay"           
 [3] "processed_transcript"               "lincRNA"                           
 [5] "polymorphic_pseudogene"             "retained_intron"                   
 [7] "antisense"                          "unprocessed_pseudogene"            
 [9] "TEC"                                "transcribed_unprocessed_pseudogene"
[11] "pseudogene"                         "processed_pseudogene"              
[13] "non_stop_decay"                     "IG_J_pseudogene"                   
[15] "IG_V_pseudogene"                    "TR_V_gene"                         
[17] "rRNA"                               "miRNA"                             
[19] "snoRNA"                             "Mt_tRNA"                           
[21] "ribozyme"                           "snRNA"                             
[23] "Mt_rRNA"                            "misc_RNA"                          
[25] "scaRNA"                             "IG_C_pseudogene"                   
[27] "sense_intronic"                     "IG_pseudogene"                     
[29] "sense_overlapping"                  "TR_J_gene"                         
[31] "IG_C_gene"                          "sRNA"                              
[33] "TR_V_pseudogene"                    "TR_D_gene"

In [467]:
# Let's focus on protein coding genes and typical ncRNA genes
sel_genes <- c('protein_coding',
  'ncRNA', 
  'lncRNA', 
  'miRNA', 'lincRNA', 'pre_miRNA',
  'scRNA', 'snRNA', 'snoRNA', 'scaRNA', 'sRNA', "piRNA",
  'misc_RNA', 
  'Mt_tRNA', 'Mt_rRNA', 'rRNA', 'rRNA_pseudogene', "tRNA",
  'processed_transcript', 'ribozyme'
)

# we would like to get unique annotation for each transcript
transcripts_annotation <- transcripts_filtered %>%
    select(Chromosome.scaffold.name, Gene.start..bp., Gene.end..bp., Strand,
          Gene.stable.ID, Gene.stable.ID,
          Transcript.stable.ID, Transcript.start..bp., Transcript.end..bp., Transcript.type,
          Gene.type, Gene.name, Gene.description) %>%
    distinct() %>%
    # Calculate TSS based on strand: positive strand uses start, negative strand uses end
    mutate(TSS = ifelse(Strand == 1, Transcript.start..bp., Transcript.end..bp.)) %>%
    # Create priority for gene types (lower number = higher priority)
    mutate(gene_type_priority = case_when(
      Transcript.type %in% sel_genes ~ match(Transcript.type, sel_genes),
      TRUE ~ 100  # All other types get lowest priority
    )) %>%
    # For each transcript ID, keep only the one with highest priority (lowest number)
    group_by(Transcript.stable.ID) %>%
    slice_min(gene_type_priority, n = 1, with_ties = FALSE) %>%
    ungroup() %>%
    # Remove the priority column as it's no longer needed
    select(-gene_type_priority) %>%
    arrange(Chromosome.scaffold.name, Gene.start..bp.)


In [468]:
cat("Number of transcripts with selected annotation: ", nrow(transcripts_annotation))

Number of transcripts with selected annotation:  65905

In [469]:
# Let's add annotations of protein_coding and ncRNA
transcripts_annotation <- transcripts_annotation %>%
        mutate(is_pcg = ifelse(Transcript.type == sel_genes[1],yes = TRUE,no = FALSE)) %>%
        mutate(is_ncrna = ifelse(Transcript.type %in% sel_genes[-1],yes = TRUE,no = FALSE))

In [470]:
tt <- transcripts_annotation %>% 
    group_by(Transcript.stable.ID) %>%
    count(Transcript.type) %>%
    filter(n > 1) 
cat("Any transcript with multiple type annotation? ",  nrow(tt) != 0 )
cat("\nAny duplicated transcript id?", any(duplicated(transcripts_filtered$Transcript.stable.ID)))
rm(tt)

Any transcript with multiple type annotation?  FALSE
Any duplicated transcript id? FALSE

In [471]:
cat("Number of ncRNA:" )
transcripts_annotation %>% filter(is_ncrna) %>% pull(Transcript.type) %>% table() %>% sort(decreasing = TRUE)
cat("Number of protein coding genes:" )
transcripts_annotation %>% filter(is_pcg) %>% pull(Transcript.type) %>% table() %>% sort(decreasing = TRUE)
cat("Number of other genes:" )
transcripts_annotation %>% filter(!is_pcg & !is_ncrna) %>% pull(Transcript.type) %>% table() %>% sort(decreasing = TRUE)

Number of ncRNA:

.
processed_transcript              lincRNA                 rRNA 
                3359                 2296                 1966 
               snRNA                miRNA               snoRNA 
                 541                  435                  247 
            misc_RNA              Mt_tRNA               scaRNA 
                  94                   22                   11 
            ribozyme                 sRNA              Mt_rRNA 
                   4                    4                    2 

Number of protein coding genes:

protein_coding 
         51259

Number of other genes:

.
                   retained_intron                          antisense 
                              3537                                892 
           nonsense_mediated_decay             unprocessed_pseudogene 
                               668                                221 
                         TR_J_gene                          TR_V_gene 
                                76                                 70 
                    sense_intronic transcribed_unprocessed_pseudogene 
                                63                                 29 
              processed_pseudogene                    IG_V_pseudogene 
                                27                                 17 
                               TEC                         pseudogene 
                                16                                 14 
            polymorphic_pseudogene                  sense_overlapping 
                                10                                 10 
    

In [472]:
# save the annotations
out_file = paste0(out_dir, species, "_", date, ".tsv")
write.table(transcripts_annotation,file = out_file, sep = "\t",quote = FALSE,col.names = TRUE,  )